# Performance Analytics
Fully runnable notebook for mutual fund performance analysis.


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from scipy import stats
import plotly.graph_objects as go

out = Path('output')
out.mkdir(exist_ok=True)

fund = pd.read_csv('01_fund_master.csv')
nav = pd.read_csv('02_nav_history.csv')
perf = pd.read_csv('07_scheme_performance.csv')
bench = pd.read_csv('10_benchmark_indices.csv')

for df in [fund, nav, perf, bench]:
    df.columns = [re.sub(r'[^a-z0-9]+','_', c.strip().lower()).strip('_') for c in df.columns]

nav['amfi_code'] = pd.to_numeric(nav['amfi_code'], errors='coerce')
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
nav['nav'] = pd.to_numeric(nav['nav'], errors='coerce')
perf['amfi_code'] = pd.to_numeric(perf['amfi_code'], errors='coerce')
bench['date'] = pd.to_datetime(bench['date'], errors='coerce')
bench['close_value'] = pd.to_numeric(bench['close_value'], errors='coerce')
for c in ['return_1yr_pct','return_3yr_pct','return_5yr_pct','expense_ratio_pct','alpha','beta','sharpe_ratio','sortino_ratio','std_dev_ann_pct','max_drawdown_pct']:
    if c in perf.columns:
        perf[c] = pd.to_numeric(perf[c], errors='coerce')

nav = nav.merge(fund[['amfi_code','scheme_name']], on='amfi_code', how='left')
nav = nav.sort_values(['amfi_code','date'])
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
nav['running_max'] = nav.groupby('amfi_code')['nav'].cummax()
nav['drawdown'] = nav['nav'] / nav['running_max'] - 1


FileNotFoundError: [Errno 2] No such file or directory: '10_benchmark_indices.csv'

## Daily return validation


In [ ]:
ret = nav['daily_return'].dropna()
summary = pd.DataFrame({'metric':['count','mean','std','min','1%','5%','50%','95%','99%','max'],'value':[ret.count(),ret.mean(),ret.std(),ret.min(),ret.quantile(.01),ret.quantile(.05),ret.quantile(.5),ret.quantile(.95),ret.quantile(.99),ret.max()]})
summary.to_csv(out/'daily_return_summary.csv', index=False)
summary


## CAGR by period


In [ ]:
rows=[]
for amfi,g in nav.groupby('amfi_code'):
    g=g.dropna(subset=['nav']).sort_values('date')
    if g.empty: continue
    scheme=g['scheme_name'].dropna().iloc[0] if g['scheme_name'].notna().any() else str(amfi)
    end_date=g['date'].max()
    for yrs in [1,3,5]:
        start_cut=end_date-pd.DateOffset(years=yrs)
        h=g[g['date']>=start_cut].copy()
        if len(h)<2: continue
        nav_start, nav_end = h['nav'].iloc[0], h['nav'].iloc[-1]
        n=(h['date'].iloc[-1]-h['date'].iloc[0]).days/365.25
        if nav_start>0 and n>0:
            cagr=(nav_end/nav_start)**(1/n)-1
            rows.append({'amfi_code':amfi,'scheme_name':scheme,'period':f'{yrs}yr','cagr':cagr})
cagr_df=pd.DataFrame(rows)
cagr_df.to_csv(out/'cagr_comparison.csv', index=False)
cagr_df.head()


## Ratios, alpha, beta, drawdown


In [ ]:
rf=0.065
ratio_rows=[]
for amfi,g in nav.groupby('amfi_code'):
    g=g.sort_values('date').dropna(subset=['nav'])
    r=g['daily_return'].dropna()
    if len(r)<20: continue
    mean_r=r.mean()*252
    std_r=r.std()*np.sqrt(252)
    downside=r[r<0]
    dstd=downside.std()*np.sqrt(252) if len(downside)>1 else np.nan
    ratio_rows.append({'amfi_code':amfi,'scheme_name':g['scheme_name'].dropna().iloc[0] if g['scheme_name'].notna().any() else str(amfi),'sharpe_calc':(mean_r-rf)/std_r if std_r and not np.isnan(std_r) else np.nan,'sortino_calc':(mean_r-rf)/dstd if pd.notna(dstd) and dstd!=0 else np.nan})
ratios=pd.DataFrame(ratio_rows)

n100=bench[bench['index_name'].astype(str).str.upper().eq('NIFTY100')].sort_values('date').copy()
n100['bench_ret']=n100['close_value'].pct_change()
n100=n100[['date','bench_ret']].dropna()
alpha_rows=[]
mdd_rows=[]
for amfi,g in nav.groupby('amfi_code'):
    gg=g[['date','daily_return']].dropna().merge(n100, on='date', how='inner')
    if len(gg)>=30:
        slope, intercept, r, p, se = stats.linregress(gg['bench_ret'], gg['daily_return'])
        alpha_rows.append({'amfi_code':amfi,'scheme_name':g['scheme_name'].dropna().iloc[0] if g['scheme_name'].notna().any() else str(amfi),'alpha_annual':intercept*252,'beta':slope,'obs':len(gg)})
    dd=g.sort_values('date').dropna(subset=['nav']).copy()
    if len(dd)>=2:
        draw=dd['nav']/dd['nav'].cummax()-1
        trough_idx=draw.idxmin()
        trough_date=dd.loc[trough_idx,'date']
        peak_date=dd.loc[dd.loc[:trough_idx,'nav'].idxmax(),'date']
        mdd_rows.append({'amfi_code':amfi,'scheme_name':g['scheme_name'].dropna().iloc[0] if g['scheme_name'].notna().any() else str(amfi),'max_drawdown':draw.min(),'peak_date':peak_date,'trough_date':trough_date})
alpha_beta=pd.DataFrame(alpha_rows)
mdd_df=pd.DataFrame(mdd_rows)
alpha_beta.to_csv(out/'alpha_beta.csv', index=False)
alpha_beta.head()


## Scorecard


In [ ]:
score=perf[['amfi_code','scheme_name','expense_ratio_pct']].drop_duplicates().merge(cagr_df[cagr_df['period']=='3yr'][['amfi_code','cagr']], on='amfi_code', how='left').merge(ratios[['amfi_code','sharpe_calc']], on='amfi_code', how='left').merge(alpha_beta[['amfi_code','alpha_annual']], on='amfi_code', how='left').merge(mdd_df[['amfi_code','max_drawdown']], on='amfi_code', how='left')
score['cagr_rank']=score['cagr'].rank(ascending=False, method='min')
score['sharpe_rank']=score['sharpe_calc'].rank(ascending=False, method='min')
score['alpha_rank']=score['alpha_annual'].rank(ascending=False, method='min')
score['expense_rank']=score['expense_ratio_pct'].rank(ascending=True, method='min')
score['mdd_rank']=score['max_drawdown'].rank(ascending=False, method='min')
n=len(score)
score['fund_score']=100*(0.30*(1-(score['cagr_rank']-1)/(n-1))+0.25*(1-(score['sharpe_rank']-1)/(n-1))+0.20*(1-(score['alpha_rank']-1)/(n-1))+0.15*(1-(score['expense_rank']-1)/(n-1))+0.10*(1-(score['mdd_rank']-1)/(n-1)))
score=score.sort_values('fund_score', ascending=False)
score.to_csv(out/'fund_scorecard.csv', index=False)
score.head(10)


## Benchmark chart


In [ ]:
top5=score.head(5)['amfi_code'].tolist()
n50=bench[bench['index_name'].astype(str).str.upper().eq('NIFTY50')].sort_values('date')[['date','close_value']].copy()
n100=bench[bench['index_name'].astype(str).str.upper().eq('NIFTY100')].sort_values('date')[['date','close_value']].copy()
common=n50.merge(n100, on='date', suffixes=('_n50','_n100'))
common['n50_idx']=common['close_value_n50']/common['close_value_n50'].iloc[0]*100
common['n100_idx']=common['close_value_n100']/common['close_value_n100'].iloc[0]*100
fig=go.Figure()
for amfi in top5:
    g=nav[nav['amfi_code']==amfi].sort_values('date')[['date','nav','scheme_name']].dropna()
    g=g[g['date']>=g['date'].max()-pd.DateOffset(years=3)]
    if len(g)<30: continue
    g['idx']=g['nav']/g['nav'].iloc[0]*100
    fig.add_trace(go.Scatter(x=g['date'], y=g['idx'], mode='lines', name=g['scheme_name'].iloc[0]))
fig.add_trace(go.Scatter(x=common['date'], y=common['n50_idx'], mode='lines', name='NIFTY50', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=common['date'], y=common['n100_idx'], mode='lines', name='NIFTY100', line=dict(dash='dot')))
fig.update_layout(title='Top 5 funds vs NIFTY50 and NIFTY100 (3 years)', xaxis_title='Date', yaxis_title='Indexed value', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5))
fig.write_image(out/'benchmark_comparison.png')
fig


## Tracking error


In [ ]:
te=[]
bench3=common[['date','n50_idx','n100_idx']].copy()
for amfi in top5:
    g=nav[nav['amfi_code']==amfi].sort_values('date')[['date','nav','scheme_name']].dropna()
    g=g[g['date']>=g['date'].max()-pd.DateOffset(years=3)].copy()
    g['ret']=g['nav'].pct_change()
    merged=g[['date','ret']].merge(bench[bench['index_name'].astype(str).str.upper().eq('NIFTY50')][['date','close_value']].sort_values('date').assign(nifty50=lambda x: x['close_value'].pct_change())[ ['date','nifty50']], on='date', how='inner').merge(bench[bench['index_name'].astype(str).str.upper().eq('NIFTY100')][['date','close_value']].sort_values('date').assign(nifty100=lambda x: x['close_value'].pct_change())[ ['date','nifty100']], on='date', how='inner').dropna()
    te.append({'amfi_code':amfi,'scheme_name':g['scheme_name'].iloc[0] if len(g) else str(amfi),'tracking_error_nifty50':(merged['ret']-merged['nifty50']).std()*np.sqrt(252),'tracking_error_nifty100':(merged['ret']-merged['nifty100']).std()*np.sqrt(252)})
tr=pd.DataFrame(te)
tr.to_csv(out/'tracking_error.csv', index=False)
tr


## Done
Generated files are saved in `output/`.
